In [4]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os
import sys


sys.path.append(os.path.join(os.getcwd(), '..'))

print("Python libraries loaded successfully")

Python libraries loaded successfully


In [5]:
#Load raw data 

df = pd.read_csv("../data/raw/creditcard.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [6]:
# 1. log1p transform on Amount
df['Amount_log'] = np.log1p(df['Amount'])

# 2. Normalize Time to [0, 1]
df['Time_norm'] = (df['Time'] - df['Time'].min()) / (df['Time'].max() - df['Time'].min())

# 3. Drop original Amount and Time
df = df.drop(columns=['Amount', 'Time'])

print("New columns:", list(df.columns))
print("Shape:", df.shape)

New columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_log', 'Time_norm']
Shape: (284807, 31)


In [7]:
# Separate features and target
X = df.drop(columns=['Class'])
y = df['Class']

# Split - stratify keeps class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train fraud %: {y_train.mean()*100:.3f}%")
print(f"y_test  fraud %: {y_test.mean()*100:.3f}%")

X_train: (227845, 30)
X_test:  (56962, 30)
y_train fraud %: 0.173%
y_test  fraud %: 0.172%


In [8]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - X_train: {X_train.shape}")
print(f"After SMOTE  - X_train: {X_train_resampled.shape}")
print(f"Before SMOTE - fraud count: {y_train.sum()}")
print(f"After SMOTE  - fraud count: {y_train_resampled.sum()}")
print(f"After SMOTE  - fraud %: {y_train_resampled.mean()*100:.1f}%")

Before SMOTE - X_train: (227845, 30)
After SMOTE  - X_train: (454902, 30)
Before SMOTE - fraud count: 394
After SMOTE  - fraud count: 227451
After SMOTE  - fraud %: 50.0%


In [9]:
import os

# Create directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save all splits
X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_resampled.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Saved:")
print(f"  X_train: {X_train_resampled.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train_resampled.shape}")
print(f"  y_test:  {y_test.shape}")

Saved:
  X_train: (454902, 30)
  X_test:  (56962, 30)
  y_train: (454902,)
  y_test:  (56962,)
